# Fase 1B — Preprocesamiento para Transformers (IroSvA)

Este notebook concentra el **preprocesamiento y la auditoría común de datos** para los modelos Transformer.

Modelos objetivo actuales:

- **BETO cased** — `dccuchile/bert-base-spanish-wwm-cased`
- **DistilBETO uncased** — `dccuchile/distilbert-base-spanish-uncased`

También se conserva soporte para:

- **RoBERTuito uncased** — `pysentimiento/robertuito-base-uncased`

## Principios

- Se parte siempre de `MESSAGE` original.
- No se utiliza `MESSAGE_CLEAN`, stemming ni lematización del pipeline de Machine Learning.
- Se mantienen stopwords, puntuación y estructura de la oración.
- BETO y DistilBETO compararán posteriormente dos alternativas de entrada:
  - `BASE`: normalización mínima.
  - `SOCIAL`: normalización específica de redes sociales.
- La descontaminación Train ↔ Test se define **una sola vez en este notebook**.
- La regla de descontaminación utiliza únicamente el contenido textual y las secuencias tokenizadas; **las etiquetas del test no intervienen en la decisión**.
- Se elimina únicamente del **train** cualquier fila cuya entrada coincida con alguna entrada del test bajo las representaciones auditadas.
- El test oficial se mantiene intacto.
- La construcción de `GROUP_ID`, `StratifiedGroupKFold`, Nested CV y la optimización de hiperparámetros permanecen en los notebooks de BETO y DistilBETO.


## 1. Librerías y configuración

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import emoji

try:
    from transformers import AutoTokenizer
    TRANSFORMERS_DISPONIBLE = True
except ImportError:
    AutoTokenizer = None
    TRANSFORMERS_DISPONIBLE = False

try:
    from pysentimiento.preprocessing import preprocess_tweet
    PYSENTIMIENTO_DISPONIBLE = True
except ImportError:
    preprocess_tweet = None
    PYSENTIMIENTO_DISPONIBLE = False

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "transformers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
}

ROBERTUITO_ALTERNATIVO = "pysentimiento/robertuito-base-deacc"

print("transformers disponible:", TRANSFORMERS_DISPONIBLE)
print("pysentimiento disponible:", PYSENTIMIENTO_DISPONIBLE)
print()
for modelo, checkpoint in CHECKPOINTS.items():
    print(f"{modelo:12} -> {checkpoint}")
print(f"{'RoBERTuito alt.':12} -> {ROBERTUITO_ALTERNATIVO}")


transformers disponible: True
pysentimiento disponible: False

BETO         -> dccuchile/bert-base-spanish-wwm-cased
DistilBETO   -> dccuchile/distilbert-base-spanish-uncased
RoBERTuito   -> pysentimiento/robertuito-base-uncased
RoBERTuito alt. -> pysentimiento/robertuito-base-deacc


## 2. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar duplicados se verifica:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Solo los duplicados exactos de texto se deduplican después de la auditoría, manteniendo una única copia.  
Esto replica el criterio utilizado en el preprocessing de ML para mantener ambos experimentos uniformizados.


In [2]:
def auditar_duplicados_train(df, variante):
    grupos = (
        df.groupby("MESSAGE", dropna=False)["IS_IRONIC"]
        .agg(["size", "nunique"])
        .reset_index()
    )

    repetidos = grupos[
        grupos["size"] > 1
    ].copy()

    conflictos = repetidos[
        repetidos["nunique"] > 1
    ].copy()

    mismos_labels = repetidos[
        repetidos["nunique"] == 1
    ].copy()

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismos_labels),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(
            conflictos["MESSAGE"].tolist()
        ),
        "conflictos_resumen": conflictos,
    }


def cargar_train():
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in ["mx", "es", "cu"]:
        df = pd.read_csv(
            DATA_DIR / f"irosva.{variante}.training.csv"
        )
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(
            df,
            variante,
        )

        mensajes_conflicto = info[
            "mensajes_conflicto"
        ]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[
            df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        if not df_conflictos.empty:
            df_conflictos[
                "MOTIVO_EXCLUSION"
            ] = "mismo_texto_etiquetas_contradictorias"

            conflictos_detalle.append(
                df_conflictos
            )

            print(
                f"\n{variante.upper()} — conflictos de etiqueta:"
            )

            display(
                df_conflictos[
                    [
                        "ID",
                        "MESSAGE",
                        "IS_IRONIC",
                        "VARIANTE",
                    ]
                ].sort_values("MESSAGE")
            )

        # 1. Eliminar completamente textos con etiquetas contradictorias.
        df_sin_conflictos = df[
            ~df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        # 2. En duplicados restantes, las etiquetas son iguales:
        #    conservar una única copia.
        antes_dedup = len(df_sin_conflictos)

        df_limpio = (
            df_sin_conflictos
            .drop_duplicates(
                subset="MESSAGE",
                keep="first",
            )
            .reset_index(drop=True)
        )

        filas_dedup_mismo_label = (
            antes_dedup
            - len(df_limpio)
        )

        auditoria.append({
            "variante": variante,
            "n_original": info["n_original"],
            "textos_repetidos": info["n_textos_repetidos"],
            "conflictos_texto": info["n_conflictos_etiqueta"],
            "filas_retiradas_conflicto": len(df_conflictos),
            "filas_duplicadas_misma_etiqueta_retiradas": filas_dedup_mismo_label,
            "n_tras_control_duplicados": len(df_limpio),
        })

        print(
            f"{variante}: "
            f"{info['n_original']} originales | "
            f"{len(df_conflictos)} filas retiradas por conflicto | "
            f"{filas_dedup_mismo_label} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )

        frames.append(
            df_limpio
        )

    df_train = pd.concat(
        frames,
        ignore_index=True,
    )

    df_auditoria = pd.DataFrame(
        auditoria
    )

    if conflictos_detalle:
        df_conflictos_train = pd.concat(
            conflictos_detalle,
            ignore_index=True,
        )
    else:
        df_conflictos_train = pd.DataFrame()

    return (
        df_train,
        df_auditoria,
        df_conflictos_train,
    )


(
    df_train,
    df_auditoria_train,
    df_conflictos_train,
) = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(
    "\nTrain tras control de duplicados/conflictos:",
    len(df_train),
)

print(
    df_train["VARIANTE"]
    .value_counts()
    .sort_index()
)

print("\nDistribución de clases:")
print(
    df_train
    .groupby("VARIANTE")["IS_IRONIC"]
    .value_counts()
)


mx: 2400 originales | 0 filas retiradas por conflicto | 1 duplicados de misma etiqueta retirados | 2399 restantes

ES — conflictos de etiqueta:


,ID,MESSAGE,IS_IRONIC,VARIANTE
762,3bb2e28f5c0f1a0dd7fbcbed779ff957,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,1,es
916,5353523b4ac97aff03c46cb3d11d4d6a,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,0,es


es: 2400 originales | 2 filas retiradas por conflicto | 1 duplicados de misma etiqueta retirados | 2397 restantes
cu: 2400 originales | 0 filas retiradas por conflicto | 0 duplicados de misma etiqueta retirados | 2400 restantes

AUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS


,variante,n_original,textos_repetidos,conflictos_texto,filas_retiradas_conflicto,filas_duplicadas_misma_etiqueta_retiradas,n_tras_control_duplicados
0,mx,2400,1,0,0,1,2399
1,es,2400,2,1,2,1,2397
2,cu,2400,0,0,0,0,2400



Train tras control de duplicados/conflictos: 7196
VARIANTE
cu    2400
es    2397
mx    2399
Name: count, dtype: int64

Distribución de clases:
VARIANTE  IS_IRONIC
cu        0            1600
          1             800
es        0            1598
          1             799
mx        0            1599
          1             800
Name: count, dtype: int64


## 3. Carga del test oficial

El test se carga para:

1. generar sus representaciones finales;
2. realizar la auditoría Train ↔ Test;
3. exportar el archivo que se utilizará en la evaluación final.

La **descontaminación no utiliza `IS_IRONIC` del test**.  
Las etiquetas se conservan únicamente porque forman parte del archivo final de evaluación.

El test no participa en:

- elección BASE/SOCIAL;
- selección de `max_length`;
- selección de hiperparámetros;
- selección de checkpoint;
- Nested Cross-Validation.


In [3]:
def cargar_test(variante):
    df_texto = pd.read_csv(
        DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv"
    )
    df_truth = pd.read_csv(
        DATA_DIR / f"irosva.{variante}.test.truth.csv"
    )

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(
        df_truth[["ID", "IS_IRONIC"]],
        on="ID",
        how="inner",
    )

    df["VARIANTE"] = variante
    return df


df_test = pd.concat(
    [cargar_test(v) for v in ["mx", "es", "cu"]],
    ignore_index=True,
)

print("Test total:", len(df_test))
print(df_test["VARIANTE"].value_counts().sort_index())


Test total: 1800
VARIANTE
cu    600
es    600
mx    600
Name: count, dtype: int64


## 4. Auditoría preliminar Train ↔ Test — texto original

Antes de generar BASE/SOCIAL se registra si existen coincidencias exactas de `MESSAGE`.

**En este bloque no se elimina ninguna fila.**  
La exclusión se realiza una sola vez después de generar todas las representaciones relevantes para BETO y DistilBETO.

Esto evita tener varias políticas de descontaminación repartidas entre notebooks.


In [4]:
def auditar_train_test_original(df_train, df_test):
    filas = []
    coincidencias_detalle = {}

    for variante in ["mx", "es", "cu"]:
        tr = df_train[
            df_train["VARIANTE"] == variante
        ].copy()

        te = df_test[
            df_test["VARIANTE"] == variante
        ].copy()

        dups_test = int(
            te.duplicated(subset="MESSAGE").sum()
        )

        mensajes_train = set(
            tr["MESSAGE"].fillna("").astype(str)
        )

        mensajes_test = set(
            te["MESSAGE"].fillna("").astype(str)
        )

        comunes = mensajes_train & mensajes_test

        # La etiqueta del test NO interviene en esta auditoría.
        detalle_train = tr[
            tr["MESSAGE"].fillna("").astype(str).isin(comunes)
        ][
            ["ID", "MESSAGE", "IS_IRONIC"]
        ].rename(
            columns={
                "ID": "ID_TRAIN",
                "IS_IRONIC": "LABEL_TRAIN",
            }
        )

        detalle_test = te[
            te["MESSAGE"].fillna("").astype(str).isin(comunes)
        ][
            ["ID", "MESSAGE"]
        ].rename(
            columns={
                "ID": "ID_TEST",
            }
        )

        detalle = detalle_train.merge(
            detalle_test,
            on="MESSAGE",
            how="inner",
        )

        coincidencias_detalle[variante] = detalle

        filas.append({
            "variante": variante,
            "train_n": len(tr),
            "test_n": len(te),
            "duplicados_test_message": dups_test,
            "coincidencias_originales": len(comunes),
            "filas_train_implicadas": len(detalle_train),
        })

    return pd.DataFrame(filas), coincidencias_detalle


df_auditoria_train_test, coincidencias_train_test = (
    auditar_train_test_original(
        df_train,
        df_test,
    )
)

print("AUDITORÍA PRELIMINAR TRAIN ↔ TEST — MESSAGE ORIGINAL")
display(df_auditoria_train_test)

for variante in ["mx", "es", "cu"]:
    detalle = coincidencias_train_test[variante]

    if not detalle.empty:
        print(
            f"[AUDIT][Transformers][{variante.upper()}] "
            f"{len(detalle)} pares con MESSAGE original coincidente."
        )
        display(detalle.head(10))

print(
    "\nLa exclusión definitiva se realizará después de generar "
    "BASE/SOCIAL y las claves tokenizadas."
)


AUDITORÍA PRELIMINAR TRAIN ↔ TEST — MESSAGE ORIGINAL


,variante,train_n,test_n,duplicados_test_message,coincidencias_originales,filas_train_implicadas
0,mx,2399,600,0,2,2
1,es,2397,600,0,1,1
2,cu,2400,600,0,0,0


[AUDIT][Transformers][MX] 2 pares con MESSAGE original coincidente.


,ID_TRAIN,MESSAGE,LABEL_TRAIN,ID_TEST
0,910b1a08a43e051796edf541480ea2ae,No lo puede evitar teacher. Nacio perversa de...,0,910b1a08a43e051796edf541480ea2ae
1,61a69eb1aa7c10192d9b2e7c06927ab2,Qué falta de seriedad,0,61a69eb1aa7c10192d9b2e7c06927ab2


[AUDIT][Transformers][ES] 1 pares con MESSAGE original coincidente.


,ID_TRAIN,MESSAGE,LABEL_TRAIN,ID_TEST
0,515135acbbd86f0d07f653a471ad2b23,#YoConAlbert Einstein,1,807327f4adead1d48fc2317a1f70588b



La exclusión definitiva se realizará después de generar BASE/SOCIAL y las claves tokenizadas.


## 5. Features lingüísticas opcionales

Se conservan las mismas 9 features del experimento ML, calculadas **desde el texto original**.

No se incorporarán en el primer baseline Transformer. Se guardan para un experimento posterior:

**Transformer puro vs Transformer + features lingüísticas**.


In [5]:
NEGACIONES = {
    "no", "nunca", "jamas", "tampoco",
    "nada", "nadie",
    "ningun", "ninguno", "ninguna", "ningunos", "ningunas",
    "ni", "sino", "sin",
}

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]


def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def extraer_features_linguisticos(texto):
    texto_str = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras = re.findall(r"\b\w+\b", texto_norm)

    n_exc = texto_str.count("!") + texto_str.count("¡")
    n_int = texto_str.count("?") + texto_str.count("¿")
    n_may = len(
        re.findall(r"\b[A-ZÁÉÍÓÚÜÑ]{3,}\b", texto_str)
    )
    n_emo = sum(
        1 for c in texto_str
        if c in emoji.EMOJI_DATA
    )
    n_ris = len(
        re.findall(
            r"\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b",
            texto_norm,
        )
    )
    n_neg = sum(
        1 for p in palabras
        if p in NEGACIONES
    )
    n_elo = len(
        re.findall(r"(.)\1{2,}", texto_norm)
    )
    n_com = len(
        re.findall(r'["\'‘’“”«»]', texto_str)
    )
    n_pun = texto_str.count("...") + texto_str.count("…")

    return [
        n_exc, n_int, n_may, n_emo, n_ris,
        n_neg, n_elo, n_com, n_pun,
    ]


for nombre, df in [("Train", df_train), ("Test", df_test)]:
    features = df["MESSAGE"].fillna("").apply(
        extraer_features_linguisticos
    ).tolist()

    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in features]

    print(f"{nombre}: features lingüísticas calculadas.")


Train: features lingüísticas calculadas.
Test: features lingüísticas calculadas.


## 6. Preprocessing BASE para BETO y DistilBETO

Esta versión realiza una limpieza mínima:

- Unicode y espacios normalizados.
- URL → `url`.
- mención → `@usuario`.
- elimina únicamente marcadores aislados `RT` / `FAV`.

Se conservan:

- mayúsculas;
- tildes;
- puntuación;
- `¿?` / `¡!`;
- hashtags;
- emojis;
- elongaciones;
- stopwords.

**DistilBETO es uncased:** no hace falta convertir manualmente el texto a minúsculas; se deja esa responsabilidad a su tokenizer oficial.


In [6]:
URL_RE = re.compile(
    r"(?:https?://\S+|www\.\S+)",
    flags=re.IGNORECASE,
)

MENTION_RE = re.compile(r"(?<!\w)@\w+")

RT_FAV_RE = re.compile(
    r"\b(?:RT|FAV)\b",
    flags=re.IGNORECASE,
)


def normalizar_unicode_y_espacios(texto):
    texto = unicodedata.normalize(
        "NFKC",
        str(texto),
    )

    texto = "".join(
        c for c in texto
        if unicodedata.category(c)[0] != "C"
        or c in "\n\t"
    )

    return re.sub(r"\s+", " ", texto).strip()


def preprocesar_transformer_base(texto):
    texto = normalizar_unicode_y_espacios(texto)

    texto = URL_RE.sub(
        " url ",
        texto,
    )

    texto = MENTION_RE.sub(
        " @usuario ",
        texto,
    )

    texto = RT_FAV_RE.sub(
        " ",
        texto,
    )

    return re.sub(r"\s+", " ", texto).strip()


## 7. Preprocessing SOCIAL para BETO y DistilBETO

La literatura sobre Twitter en español deja abiertas algunas decisiones para BETO/DistilBETO.

Por eso esta segunda variante permite comprobar posteriormente, mediante validación, si ayuda:

- limitar elongaciones a 3 caracteres;
- transformar hashtags en contenido textual;
- convertir emojis a descripción textual en español.

La puntuación y las stopwords se siguen conservando.


In [7]:
def limitar_elongaciones(
    texto,
    max_repeticiones=3,
):
    patron = re.compile(
        r"(.)\1{" + str(max_repeticiones) + r",}",
        flags=re.IGNORECASE,
    )

    return patron.sub(
        lambda m: m.group(1) * max_repeticiones,
        texto,
    )


def separar_camel_case(token):
    token = token.replace("_", " ")

    token = re.sub(
        r"(?<=[a-záéíóúüñ])(?=[A-ZÁÉÍÓÚÜÑ])",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])(?=\d)",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=\d)(?=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])",
        " ",
        token,
    )

    return re.sub(
        r"\s+",
        " ",
        token,
    ).strip()


def normalizar_hashtags(texto):
    def reemplazar(match):
        contenido = separar_camel_case(
            match.group(1)
        )

        return f" hashtag {contenido} "

    return re.sub(
        r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)",
        reemplazar,
        texto,
    )


def emojis_a_texto_es(texto):
    # Ejemplo aproximado:
    # 🙄 -> :cara_con_ojos_en_blanco:
    texto = emoji.demojize(
        texto,
        language="es",
    )

    patron = re.compile(
        r":([^:\s]+):"
    )

    def reemplazar(match):
        descripcion = (
            match.group(1)
            .replace("_", " ")
        )

        return (
            f" emoji {descripcion} emoji "
        )

    return patron.sub(
        reemplazar,
        texto,
    )


def preprocesar_transformer_social(texto):
    texto = preprocesar_transformer_base(
        texto
    )

    texto = limitar_elongaciones(
        texto,
        max_repeticiones=3,
    )

    texto = normalizar_hashtags(
        texto
    )

    texto = emojis_a_texto_es(
        texto
    )

    return re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()


## 8. RoBERTuito — preprocessing oficial

RoBERTuito fue entrenado específicamente para redes sociales en español.

Se utiliza directamente:

```python
pysentimiento.preprocessing.preprocess_tweet
```

en lugar de replicar manualmente su lógica.


In [8]:
def preprocesar_robertuito(texto):
    if not PYSENTIMIENTO_DISPONIBLE:
        raise ImportError(
            "Falta pysentimiento. Instala con "
            "`pip install pysentimiento` o "
            "`uv add pysentimiento`."
        )

    return preprocess_tweet(
        str(texto),
        lang="es",
        shorten=3,
    )


## 9. Generación de textos Transformer

In [9]:
for nombre, df in [
    ("Train", df_train),
    ("Test", df_test),
]:
    print(f"Procesando {nombre}...")

    df["MESSAGE_TRANSFORMER_BASE"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_base)
    )

    df["MESSAGE_TRANSFORMER_SOCIAL"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_social)
    )

    if PYSENTIMIENTO_DISPONIBLE:
        df["MESSAGE_ROBERTUITO"] = (
            df["MESSAGE"]
            .fillna("")
            .astype(str)
            .apply(preprocesar_robertuito)
        )
    else:
        df["MESSAGE_ROBERTUITO"] = pd.NA
        print(
            "  AVISO: RoBERTuito no se procesó "
            "porque falta pysentimiento."
        )

print("\nEjemplos por variante:")

for variante in ["mx", "es", "cu"]:
    row = (
        df_train[
            df_train["VARIANTE"] == variante
        ]
        .iloc[0]
    )

    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:")
    print(row["MESSAGE"])

    print("\nBASE:")
    print(row["MESSAGE_TRANSFORMER_BASE"])

    print("\nSOCIAL:")
    print(row["MESSAGE_TRANSFORMER_SOCIAL"])

    if PYSENTIMIENTO_DISPONIBLE:
        print("\nROBERTUITO:")
        print(row["MESSAGE_ROBERTUITO"])


Procesando Train...
  AVISO: RoBERTuito no se procesó porque falta pysentimiento.
Procesando Test...
  AVISO: RoBERTuito no se procesó porque falta pysentimiento.

Ejemplos por variante:

--- MX ---
ORIGINAL:
Rica económicamente, pero muy pobre en objetividad.

BASE:
Rica económicamente, pero muy pobre en objetividad.

SOCIAL:
Rica económicamente, pero muy pobre en objetividad.

--- ES ---
ORIGINAL:
@ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣

BASE:
@usuario @usuario @usuario En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣

SOCIAL:
@usuario @usuario @usuario En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca emoji cara revolviéndose de la risa emoji emoji cara revolviéndose de la risa emoji

--- CU ---
ORIGINAL:
magnifico

BASE:
magnifico

SOCIAL:
magnifico


## 10. Auditoría y descontaminación común BETO + DistilBETO

Este bloque aplica **una única política de descontaminación** antes de los experimentos.

Se audita Train ↔ Test, por variante dialectal, usando:

1. `MESSAGE` original;
2. `MESSAGE_TRANSFORMER_BASE`;
3. `MESSAGE_TRANSFORMER_SOCIAL`;
4. secuencia tokenizada BETO-BASE;
5. secuencia tokenizada BETO-SOCIAL;
6. secuencia tokenizada DistilBETO-BASE;
7. secuencia tokenizada DistilBETO-SOCIAL.

### Regla

Si una fila del train coincide con alguna entrada del test en **cualquiera** de estas representaciones, se retira únicamente del train.

Se utiliza la **unión** de las coincidencias para que BETO y DistilBETO partan del mismo train final.

- El test permanece intacto.
- Las etiquetas del test no se utilizan para decidir exclusiones.
- `max_length=128` se usa porque es la longitud fijada para los experimentos Transformer.
- La auditoría compara la secuencia que realmente recibe cada tokenizer, incluyendo truncamiento y tokens especiales.

Los `GROUP_ID` no se construyen aquí. Cada modelo los genera posteriormente con su propio tokenizer para impedir leakage entre folds del Nested CV.


In [10]:
# ============================================================
# AUDITORÍA / DESCONTAMINACIÓN COMÚN DE TRANSFORMERS
# ============================================================

MAX_LENGTH_AUDITORIA = 128
BATCH_TOKENIZACION_AUDITORIA = 256
MODELOS_AUDITORIA = ["BETO", "DistilBETO"]


def cargar_tokenizers_auditoria():
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "La auditoría común requiere `transformers`. "
            "Instala la dependencia antes de continuar."
        )

    return {
        modelo: AutoTokenizer.from_pretrained(
            CHECKPOINTS[modelo],
            use_fast=True,
        )
        for modelo in MODELOS_AUDITORIA
    }


def tokenizar_claves(
    tokenizer,
    textos,
    max_length=128,
    batch_size=256,
):
    """Devuelve una clave comparable por cada secuencia realmente recibida."""
    valores = (
        pd.Series(textos)
        .fillna("")
        .astype(str)
        .tolist()
    )

    claves = []

    for inicio in range(0, len(valores), batch_size):
        lote = valores[
            inicio:inicio + batch_size
        ]

        enc = tokenizer(
            lote,
            add_special_tokens=True,
            truncation=True,
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )

        claves.extend(
            tuple(int(x) for x in ids)
            for ids in enc["input_ids"]
        )

    return claves


def preparar_claves_auditoria(df, tokenizers):
    trabajo = df.copy()

    trabajo["__KEY_ORIGINAL"] = (
        trabajo["MESSAGE"]
        .fillna("")
        .astype(str)
    )

    trabajo["__KEY_BASE"] = (
        trabajo["MESSAGE_TRANSFORMER_BASE"]
        .fillna("")
        .astype(str)
    )

    trabajo["__KEY_SOCIAL"] = (
        trabajo["MESSAGE_TRANSFORMER_SOCIAL"]
        .fillna("")
        .astype(str)
    )

    for modelo, tokenizer in tokenizers.items():
        nombre = modelo.upper()

        trabajo[f"__TOK_{nombre}_BASE"] = tokenizar_claves(
            tokenizer,
            trabajo["MESSAGE_TRANSFORMER_BASE"],
            max_length=MAX_LENGTH_AUDITORIA,
            batch_size=BATCH_TOKENIZACION_AUDITORIA,
        )

        trabajo[f"__TOK_{nombre}_SOCIAL"] = tokenizar_claves(
            tokenizer,
            trabajo["MESSAGE_TRANSFORMER_SOCIAL"],
            max_length=MAX_LENGTH_AUDITORIA,
            batch_size=BATCH_TOKENIZACION_AUDITORIA,
        )

    return trabajo


COLUMNAS_AUDITORIA = {
    "__KEY_ORIGINAL": "original",
    "__KEY_BASE": "base",
    "__KEY_SOCIAL": "social",
    "__TOK_BETO_BASE": "beto_base_tokenizado",
    "__TOK_BETO_SOCIAL": "beto_social_tokenizado",
    "__TOK_DISTILBETO_BASE": "distilbeto_base_tokenizado",
    "__TOK_DISTILBETO_SOCIAL": "distilbeto_social_tokenizado",
}


print("=" * 78)
print("AUDITORÍA / DESCONTAMINACIÓN COMÚN — BETO + DISTILBETO")
print("=" * 78)
print(
    f"max_length={MAX_LENGTH_AUDITORIA} | "
    f"modelos={', '.join(MODELOS_AUDITORIA)}"
)

tokenizers_auditoria = cargar_tokenizers_auditoria()

print("\nGenerando claves de train...")
train_audit = preparar_claves_auditoria(
    df_train,
    tokenizers_auditoria,
)

print("Generando claves de test...")
test_audit = preparar_claves_auditoria(
    df_test,
    tokenizers_auditoria,
)

indices_excluir = set()
detalles_exclusion = []
resumen_descontaminacion = []

for variante in ["mx", "es", "cu"]:
    tr_idx = train_audit.index[
        train_audit["VARIANTE"] == variante
    ].tolist()

    te = test_audit[
        test_audit["VARIANTE"] == variante
    ]

    motivos_por_indice = {
        idx: []
        for idx in tr_idx
    }

    conteos = {}

    for columna, motivo in COLUMNAS_AUDITORIA.items():
        claves_test = set(
            te[columna].tolist()
        )

        mask = train_audit.loc[
            tr_idx,
            columna,
        ].isin(claves_test)

        idx_coincidentes = mask.index[
            mask
        ].tolist()

        conteos[motivo] = len(idx_coincidentes)

        for idx in idx_coincidentes:
            motivos_por_indice[idx].append(
                motivo
            )
            indices_excluir.add(idx)

    idx_union = [
        idx
        for idx in tr_idx
        if motivos_por_indice[idx]
    ]

    if idx_union:
        detalle = train_audit.loc[
            idx_union,
            [
                "ID",
                "MESSAGE",
                "IS_IRONIC",
                "VARIANTE",
                "MESSAGE_TRANSFORMER_BASE",
                "MESSAGE_TRANSFORMER_SOCIAL",
            ],
        ].copy()

        detalle["MOTIVOS_EXCLUSION"] = [
            " | ".join(
                motivos_por_indice[idx]
            )
            for idx in idx_union
        ]

        detalles_exclusion.append(
            detalle
        )

    resumen_descontaminacion.append({
        "variante": variante,
        "train_antes": len(tr_idx),
        "test_n": len(te),
        **conteos,
        "excluidos_union": len(idx_union),
        "train_final": len(tr_idx) - len(idx_union),
    })

    print(
        f"[AUDIT][Transformers][{variante.upper()}] "
        f"train={len(tr_idx)} -> {len(tr_idx) - len(idx_union)} | "
        f"excluidos={len(idx_union)}"
    )


df_auditoria_descontaminacion = pd.DataFrame(
    resumen_descontaminacion
)

if detalles_exclusion:
    df_train_excluidos_transformers = pd.concat(
        detalles_exclusion,
        ignore_index=True,
    )
else:
    df_train_excluidos_transformers = pd.DataFrame(
        columns=[
            "ID",
            "MESSAGE",
            "IS_IRONIC",
            "VARIANTE",
            "MESSAGE_TRANSFORMER_BASE",
            "MESSAGE_TRANSFORMER_SOCIAL",
            "MOTIVOS_EXCLUSION",
        ]
    )


# ------------------------------------------------------------
# Aplicar UNA SOLA VEZ la exclusión sobre el train común.
# ------------------------------------------------------------
mask_conservar = ~train_audit.index.isin(
    indices_excluir
)

train_audit_final = train_audit.loc[
    mask_conservar
].copy()


# ------------------------------------------------------------
# Comprobación post-descontaminación.
# Ninguna clave auditada puede quedar simultáneamente en
# train final y test dentro de la misma variante.
# ------------------------------------------------------------
errores_post = []

for variante in ["mx", "es", "cu"]:
    tr = train_audit_final[
        train_audit_final["VARIANTE"] == variante
    ]

    te = test_audit[
        test_audit["VARIANTE"] == variante
    ]

    for columna, motivo in COLUMNAS_AUDITORIA.items():
        comunes = (
            set(tr[columna].tolist())
            & set(te[columna].tolist())
        )

        if comunes:
            errores_post.append(
                {
                    "variante": variante,
                    "representacion": motivo,
                    "n_claves_comunes": len(comunes),
                }
            )

if errores_post:
    df_errores_post = pd.DataFrame(
        errores_post
    )
    display(df_errores_post)

    raise RuntimeError(
        "La descontaminación no quedó completa. "
        "Revisa las claves Train ↔ Test."
    )


# Eliminar columnas temporales de auditoría.
columnas_temporales = list(
    COLUMNAS_AUDITORIA.keys()
)

df_train = (
    train_audit_final
    .drop(
        columns=columnas_temporales,
        errors="ignore",
    )
    .reset_index(drop=True)
)

print("\nRESUMEN DE DESCONTAMINACIÓN")
display(df_auditoria_descontaminacion)

print(
    "\nTrain final por variante:"
)
print(
    df_train["VARIANTE"]
    .value_counts()
    .sort_index()
)

print(
    "\nRegistros retirados en total:",
    len(df_train_excluidos_transformers),
)

if not df_train_excluidos_transformers.empty:
    print(
        "\nPrimeras exclusiones (máximo 10):"
    )
    display(
        df_train_excluidos_transformers.head(10)
    )

print(
    "\n[PRE-FLIGHT][Transformers] "
    "Descontaminación Train ↔ Test: OK"
)


AUDITORÍA / DESCONTAMINACIÓN COMÚN — BETO + DISTILBETO
max_length=128 | modelos=BETO, DistilBETO

Generando claves de train...
Generando claves de test...
[AUDIT][Transformers][MX] train=2399 -> 2397 | excluidos=2
[AUDIT][Transformers][ES] train=2397 -> 2362 | excluidos=35
[AUDIT][Transformers][CU] train=2400 -> 2399 | excluidos=1

RESUMEN DE DESCONTAMINACIÓN


,variante,train_antes,test_n,original,base,social,beto_base_tokenizado,beto_social_tokenizado,distilbeto_base_tokenizado,distilbeto_social_tokenizado,excluidos_union,train_final
0,mx,2399,600,2,2,2,2,2,2,2,2,2397
1,es,2397,600,1,35,35,35,35,35,35,35,2362
2,cu,2400,600,0,1,1,1,1,1,1,1,2399



Train final por variante:
VARIANTE
cu    2399
es    2362
mx    2397
Name: count, dtype: int64

Registros retirados en total: 38

Primeras exclusiones (máximo 10):


,ID,MESSAGE,IS_IRONIC,VARIANTE,MESSAGE_TRANSFORMER_BASE,MESSAGE_TRANSFORMER_SOCIAL,MOTIVOS_EXCLUSION
0,910b1a08a43e051796edf541480ea2ae,No lo puede evitar teacher. Nacio perversa de...,0,mx,No lo puede evitar teacher. Nacio perversa de ...,No lo puede evitar teacher. Nacio perversa de ...,original | base | social | beto_base_tokenizad...
1,61a69eb1aa7c10192d9b2e7c06927ab2,Qué falta de seriedad,0,mx,Qué falta de seriedad,Qué falta de seriedad,original | base | social | beto_base_tokenizad...
2,f3d9389e6cab34335a9651eba1f12578,"Calvo está abrasada"": la mayoría del Gobierno ...",0,es,"Calvo está abrasada"": la mayoría del Gobierno ...","Calvo está abrasada"": la mayoría del Gobierno ...",base | social | beto_base_tokenizado | beto_so...
3,aeb2e562ceb47d6cc50364e8c744529b,"Calvo está abrasada"": la mayoría del Gobierno ...",0,es,"Calvo está abrasada"": la mayoría del Gobierno ...","Calvo está abrasada"": la mayoría del Gobierno ...",base | social | beto_base_tokenizado | beto_so...
4,c47abb0386199e543c2efa6f6814b5f0,Un juez que atacó la ley de Memoria Histórica ...,0,es,Un juez que atacó la ley de Memoria Histórica ...,Un juez que atacó la ley de Memoria Histórica ...,base | social | beto_base_tokenizado | beto_so...
5,32087d6a105749cd91e55c326e392e35,► ¿Suspenden exhumación de Francisco Franco? E...,0,es,► ¿Suspenden exhumación de Francisco Franco? E...,► ¿Suspenden exhumación de Francisco Franco? E...,base | social | beto_base_tokenizado | beto_so...
6,4da39fcdafaaae42c4f627733d70c7ad,EXCLUSIVA | Un segundo juez rechaza suspender ...,0,es,EXCLUSIVA | Un segundo juez rechaza suspender ...,EXCLUSIVA | Un segundo juez rechaza suspender ...,base | social | beto_base_tokenizado | beto_so...
7,42dab181ba08f585a239bab32c00438e,La memoria histórica es una maldad de Zapatero...,0,es,La memoria histórica es una maldad de Zapatero...,La memoria histórica es una maldad de Zapatero...,base | social | beto_base_tokenizado | beto_so...
8,5a213b364e5017632dd22516d984a8ae,"En @elconfidencial: Yusti, el juez que paraliz...",0,es,"En @usuario : Yusti, el juez que paraliza la e...","En @usuario : Yusti, el juez que paraliza la e...",base | social | beto_base_tokenizado | beto_so...
9,62f94c95ee9ee36ee047b23164a6af6d,EXCLUSIVA | Un segundo juez rechaza suspender ...,0,es,EXCLUSIVA | Un segundo juez rechaza suspender ...,EXCLUSIVA | Un segundo juez rechaza suspender ...,base | social | beto_base_tokenizado | beto_so...



[PRE-FLIGHT][Transformers] Descontaminación Train ↔ Test: OK


## 11. Diagnóstico de tokenización — SOLO TRAIN FINAL

El diagnóstico se calcula **después de la descontaminación** y utiliza únicamente el train final.

Para cada tokenizer, variante dialectal y preprocessing se calcula:

- P50;
- P95;
- P99;
- máximo;
- porcentaje > 64 tokens;
- porcentaje > 128 tokens;
- tokens por palabra;
- frecuencia de `[UNK]`, cuando aplica.

El test oficial no participa en estas estadísticas.


In [11]:
def cargar_tokenizers():
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "Falta transformers. Instala con "
            "`pip install transformers` o "
            "`uv add transformers`."
        )

    tokenizers = dict(tokenizers_auditoria)

    # RoBERTuito solo se carga si se utilizará en el diagnóstico.
    if PYSENTIMIENTO_DISPONIBLE:
        tokenizers["RoBERTuito"] = AutoTokenizer.from_pretrained(
            CHECKPOINTS["RoBERTuito"]
        )

    return tokenizers


def calcular_stats_tokenizacion(
    tokenizer,
    textos,
):
    longitudes = []
    unk = 0
    total_tokens = 0
    total_palabras = 0

    unk_id = tokenizer.unk_token_id

    for texto in textos:
        texto = str(texto)

        encoded = tokenizer(
            texto,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )

        ids = encoded["input_ids"]

        longitudes.append(
            len(ids)
        )

        total_tokens += len(ids)
        total_palabras += max(
            len(texto.split()),
            1,
        )

        if unk_id is not None:
            unk += sum(
                token == unk_id
                for token in ids
            )

    arr = np.asarray(
        longitudes,
        dtype=np.int32,
    )

    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "p99": np.percentile(arr, 99),
        "max": int(arr.max()),
        "pct_gt_64": 100 * (arr > 64).mean(),
        "pct_gt_128": 100 * (arr > 128).mean(),
        "tokens_por_palabra": (
            total_tokens
            / max(total_palabras, 1)
        ),
        "unk_por_1000_tokens": (
            1000 * unk
            / max(total_tokens, 1)
        ),
    }


def diagnosticar_train():
    tokenizers = cargar_tokenizers()

    configs = [
        (
            "BETO_BASE",
            "BETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "BETO_SOCIAL",
            "BETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
        (
            "DISTILBETO_BASE",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "DISTILBETO_SOCIAL",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
    ]

    if PYSENTIMIENTO_DISPONIBLE:
        configs.append(
            (
                "ROBERTUITO",
                "RoBERTuito",
                "MESSAGE_ROBERTUITO",
            )
        )

    filas = []

    for cfg, tokenizer_name, columna in configs:
        tok = tokenizers[tokenizer_name]

        for variante in ["mx", "es", "cu"]:
            textos = (
                df_train.loc[
                    df_train["VARIANTE"] == variante,
                    columna,
                ]
                .fillna("")
                .astype(str)
                .tolist()
            )

            stats = calcular_stats_tokenizacion(
                tok,
                textos,
            )

            filas.append({
                "configuracion": cfg,
                "tokenizer": tokenizer_name,
                "variante": variante,
                **stats,
            })

    return pd.DataFrame(filas)


if TRANSFORMERS_DISPONIBLE:
    df_tokenizacion = diagnosticar_train()

    display(
        df_tokenizacion.style.format({
            "p50": "{:.1f}",
            "p95": "{:.1f}",
            "p99": "{:.1f}",
            "pct_gt_64": "{:.2f}%",
            "pct_gt_128": "{:.2f}%",
            "tokens_por_palabra": "{:.3f}",
            "unk_por_1000_tokens": "{:.3f}",
        })
    )
else:
    df_tokenizacion = None
    print(
        "Instala transformers para ejecutar "
        "el diagnóstico."
    )


,configuracion,tokenizer,variante,n,p50,p95,p99,max,pct_gt_64,pct_gt_128,tokens_por_palabra,unk_por_1000_tokens
0,BETO_BASE,BETO,mx,2397,26.0,65.0,79.0,133,5.17%,0.04%,1.498,9.488
1,BETO_BASE,BETO,es,2362,35.0,71.0,84.0,143,9.78%,0.04%,1.603,27.298
2,BETO_BASE,BETO,cu,2399,39.0,71.0,89.0,126,10.63%,0.00%,1.437,1.279
3,BETO_SOCIAL,BETO,mx,2397,27.0,68.2,91.0,193,7.01%,0.17%,1.483,2.764
4,BETO_SOCIAL,BETO,es,2362,37.0,76.9,102.4,200,13.55%,0.34%,1.538,6.830
5,BETO_SOCIAL,BETO,cu,2399,38.0,70.0,88.0,124,10.17%,0.00%,1.424,1.280
6,DISTILBETO_BASE,DistilBETO,mx,2397,24.0,62.0,71.0,82,3.25%,0.00%,1.426,9.969
7,DISTILBETO_BASE,DistilBETO,es,2362,33.0,68.0,77.0,102,7.32%,0.00%,1.516,28.853
8,DISTILBETO_BASE,DistilBETO,cu,2399,37.0,66.0,75.0,84,6.59%,0.00%,1.354,1.357
9,DISTILBETO_SOCIAL,DistilBETO,mx,2397,26.0,65.0,81.0,180,5.17%,0.08%,1.416,2.895


## 12. Exportación final por variante

Los archivos `train_transformers_*` exportados desde este punto son los **únicos train que deben consumir BETO y DistilBETO**.

Ya incorporan:

1. control de duplicados/conflictos del train;
2. representaciones BASE/SOCIAL;
3. auditoría común Train ↔ Test;
4. descontaminación común BETO + DistilBETO.

El test oficial permanece intacto.

También se exportan los archivos de auditoría para trazabilidad metodológica.


In [12]:
COLUMNAS_TEXTO = [
    "MESSAGE_TRANSFORMER_BASE",
    "MESSAGE_TRANSFORMER_SOCIAL",
    "MESSAGE_ROBERTUITO",
]

COLUMNAS_SALIDA = (
    [
        "ID",
        "TOPIC",
        "IS_IRONIC",
        "MESSAGE",
        "VARIANTE",
    ]
    + FEATURE_COLS
    + COLUMNAS_TEXTO
)

print("=" * 78)
print("EXPORTACIÓN FINAL — TRANSFORMERS")
print("=" * 78)

rutas_exportadas = {}

for split, df in [
    ("train", df_train),
    ("test", df_test),
]:
    for variante in ["mx", "es", "cu"]:
        salida = (
            df.loc[
                df["VARIANTE"] == variante,
                COLUMNAS_SALIDA,
            ]
            .reset_index(drop=True)
        )

        ruta = OUT_DIR / f"{split}_transformers_{variante}.csv"
        salida.to_csv(ruta, index=False)
        rutas_exportadas[(split, variante)] = ruta

        print(
            f"[EXPORT][{split.upper()}][{variante.upper()}] "
            f"{len(salida)} registros -> {ruta}"
        )


# ------------------------------------------------------------
# Diagnóstico de tokenización
# ------------------------------------------------------------
if df_tokenizacion is not None:
    ruta_diag = OUT_DIR / "diagnostico_tokenizacion_train.csv"
    df_tokenizacion.to_csv(ruta_diag, index=False)
    print("\n[EXPORT][AUDIT] Diagnóstico tokenización ->", ruta_diag)


# ------------------------------------------------------------
# Auditorías de calidad del dataset
# ------------------------------------------------------------
df_auditoria_train.to_csv(
    OUT_DIR / "auditoria_duplicados_train.csv",
    index=False,
)

df_auditoria_train_test.to_csv(
    OUT_DIR / "auditoria_train_test_original.csv",
    index=False,
)

df_auditoria_descontaminacion.to_csv(
    OUT_DIR / "auditoria_descontaminacion_transformers.csv",
    index=False,
)

df_train_excluidos_transformers.to_csv(
    OUT_DIR / "train_excluidos_descontaminacion_transformers.csv",
    index=False,
)

for variante, detalle in coincidencias_train_test.items():
    if not detalle.empty:
        detalle.to_csv(
            OUT_DIR / f"coincidencias_originales_train_test_{variante}.csv",
            index=False,
        )

if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(
        OUT_DIR / "conflictos_etiqueta_train.csv",
        index=False,
    )


# ------------------------------------------------------------
# Manifest simple del preprocessing final.
# BETO y DistilBETO verifican versión y cantidades.
# ------------------------------------------------------------
audit_idx = (
    df_auditoria_descontaminacion
    .set_index("variante")
)

manifest_rows = []

for variante in ["mx", "es", "cu"]:
    audit = audit_idx.loc[variante]

    manifest_rows.append({
        "pipeline_version": "transformers_comun_v1",
        "variante": variante,
        "train_antes": int(audit["train_antes"]),
        "excluidos_union": int(audit["excluidos_union"]),
        "train_final": int(audit["train_final"]),
        "test_n": int(audit["test_n"]),
        "max_length_auditoria": MAX_LENGTH_AUDITORIA,
        "checkpoint_beto": CHECKPOINTS["BETO"],
        "checkpoint_distilbeto": CHECKPOINTS["DistilBETO"],
        "criterio_descontaminacion":
            "union(original,base,social,"
            "BETO_BASE_tok,BETO_SOCIAL_tok,"
            "DistilBETO_BASE_tok,DistilBETO_SOCIAL_tok)",
        "etiquetas_test_usadas_para_exclusion": False,
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / "manifest_preprocesamiento_transformers.csv"
manifest.to_csv(manifest_path, index=False)

print("\nMANIFEST DEL PREPROCESSING FINAL")
display(
    manifest[
        [
            "variante",
            "train_antes",
            "excluidos_union",
            "train_final",
            "test_n",
        ]
    ]
)

print("\n[PREPROCESS][Transformers] EXPORTACIÓN: OK")
print("[PREPROCESS][Transformers] Manifest:", manifest_path)
print("[PREPROCESS][Transformers] Auditorías:", OUT_DIR)


EXPORTACIÓN FINAL — TRANSFORMERS
[EXPORT][TRAIN][MX] 2397 registros -> ..\data\transformers\train_transformers_mx.csv
[EXPORT][TRAIN][ES] 2362 registros -> ..\data\transformers\train_transformers_es.csv
[EXPORT][TRAIN][CU] 2399 registros -> ..\data\transformers\train_transformers_cu.csv
[EXPORT][TEST][MX] 600 registros -> ..\data\transformers\test_transformers_mx.csv
[EXPORT][TEST][ES] 600 registros -> ..\data\transformers\test_transformers_es.csv
[EXPORT][TEST][CU] 600 registros -> ..\data\transformers\test_transformers_cu.csv

[EXPORT][AUDIT] Diagnóstico tokenización -> ..\data\transformers\diagnostico_tokenizacion_train.csv

MANIFEST DEL PREPROCESSING FINAL


,variante,train_antes,excluidos_union,train_final,test_n
0,mx,2399,2,2397,600
1,es,2397,35,2362,600
2,cu,2400,1,2399,600



[PREPROCESS][Transformers] EXPORTACIÓN: OK
[PREPROCESS][Transformers] Manifest: ..\data\transformers\manifest_preprocesamiento_transformers.csv
[PREPROCESS][Transformers] Auditorías: ..\data\transformers


## 13. Uso posterior

A partir de este punto, **BETO y DistilBETO consumen exactamente los mismos archivos de train final**:

```text
data/transformers/train_transformers_mx.csv
data/transformers/train_transformers_es.csv
data/transformers/train_transformers_cu.csv
```

La procedencia de esos archivos queda congelada mediante:

```text
data/transformers/manifest_preprocesamiento_transformers.csv
```

El manifest registra, por variante:

- cantidad antes y después de la descontaminación;
- número de exclusiones;
- checkpoints utilizados en la auditoría;
- `max_length`;
- criterio de descontaminación;
- confirmación de que las etiquetas del test no participaron en la regla de exclusión.

### Qué NO debe repetirse en BETO o DistilBETO

Los notebooks de modelos ya no deben:

- eliminar filas por coincidencias Train ↔ Test;
- volver a decidir qué es contaminación;
- auditar el raw para decidir exclusiones;
- mantener cantidades de exclusión hardcodeadas.

Solo deben verificar que los archivos coinciden con el manifest.

### Qué SÍ permanece dentro de cada modelo

Cada Transformer construye de manera independiente su propio `GROUP_ID` usando:

- texto original;
- BASE;
- SOCIAL;
- secuencia BASE producida por su tokenizer;
- secuencia SOCIAL producida por su tokenizer.

Después ejecuta `StratifiedGroupKFold` 5×3.


## 14. Preprocessing definitivo utilizado

### Control de calidad del train

1. Auditar duplicados exactos de `MESSAGE`.
2. Si un mismo texto tiene etiquetas contradictorias, retirar todas sus apariciones y registrarlas.
3. En duplicados con la misma etiqueta, conservar una sola copia.

### Representaciones

#### BETO cased y DistilBETO uncased

Se generan dos entradas:

- `MESSAGE_TRANSFORMER_BASE`: Unicode/espacios + URL y usuario normalizados; conserva casing del texto fuente, acentos, puntuación, emojis, hashtags, stopwords y elongaciones.
- `MESSAGE_TRANSFORMER_SOCIAL`: BASE + elongaciones ≤3 + hashtag convertido a contenido textual + emoji convertido a descripción española.

No se aplica stemming, lematización ni eliminación de stopwords.

### Descontaminación común Train ↔ Test

Después de generar BASE/SOCIAL se auditan:

- `MESSAGE` original;
- BASE;
- SOCIAL;
- BETO(BASE) tokenizado;
- BETO(SOCIAL) tokenizado;
- DistilBETO(BASE) tokenizado;
- DistilBETO(SOCIAL) tokenizado.

La tokenización de auditoría utiliza `max_length=128`, truncamiento y tokens especiales, igual que la entrada de entrenamiento.

Se retira **únicamente del train** la unión de filas que coinciden con el test en cualquiera de estas representaciones.

Las etiquetas del test no participan en la regla de exclusión.

### Después del preprocessing

Los notebooks de BETO y DistilBETO:

1. cargan el train final;
2. construyen `GROUP_ID` de manera independiente;
3. comprueban que ningún grupo cruce train/validation;
4. ejecutan Nested CV 5×3 con `StratifiedGroupKFold`;
5. seleccionan hiperparámetros mediante F1-Macro.

De esta forma:

- **descontaminación Train ↔ Test** = responsabilidad del preprocessing;
- **prevención de leakage entre folds** = responsabilidad del notebook de cada modelo.
